In [ ]:
# ================================================================
# SCENARIO 2 — SNOWFLAKE NOTEBOOK QUERIES DATABRICKS UC TABLES
#
# Uses SCOS (Snowpark Connect) to run PySpark DataFrames against
# Databricks UC tables federated into a Snowflake catalog-linked DB.
#
# Prerequisites:
#   1. Run 03_databricks_create_uc_tables.py in Databricks.
#   2. Run 04_sf_federate_databricks_uc.sql in Snowflake.
#   3. Upload this notebook to your Snowflake workspace.
#   4. Install snowpark-connect==* in notebook packages.
#
# !! REPLACE all <PLACEHOLDER> values below before running.
# ================================================================

# !! REPLACE: Snowflake database that is NOT a catalog-linked database.
#    Used to initialize the SCOS session (CLD as default breaks resolution).
#    Must be an existing Snowflake database you have USAGE on.
SF_INIT_DB = "<SF_MANAGED_ICEBERG_DB>"   # e.g. 'MY_ICEBERG_DB'

# !! REPLACE: Snowflake catalog-linked database containing the Databricks tables
#    (created by 04_sf_federate_databricks_uc.sql)
SF_FEDERATED_DB = "<SF_FEDERATED_DB>"    # e.g. 'MY_DATABRICKS_DB'

# !! REPLACE: Databricks UC schema name in LOWERCASE
#    Must match the schema created in 03_databricks_create_uc_tables.py
DBX_SCHEMA = "<DBX_UC_SCHEMA>"           # e.g. 'my_demo_schema'

# !! REPLACE: Non-admin reader role for masking demo
#    (created by 04_sf_federate_databricks_uc.sql)
SF_READER_ROLE = "<SF_READER_ROLE>"      # e.g. 'SNOWFLAKE_READER_ROLE'

# Fully-qualified table references (lowercase schema + table for CLD)
TBL_ORDERS    = f"{SF_FEDERATED_DB}.{DBX_SCHEMA}.customer_orders"
TBL_SENSITIVE = f"{SF_FEDERATED_DB}.{DBX_SCHEMA}.sensitive_orders"

print(f"Federated DB : {SF_FEDERATED_DB}")
print(f"Schema       : {DBX_SCHEMA}")
print(f"Orders table : {TBL_ORDERS}")
print(f"Sensitive    : {TBL_SENSITIVE}")


# Scenario 2 — Snowflake SCOS Notebook: Query Databricks UC Tables
**Snowpark Connect (SCOS)**: PySpark DataFrames executing on Snowflake's engine.

```
Snowflake Notebook  →  snowpark_connect.init_spark_session()
                    →  Snowflake SQL engine
                    →  <SF_FEDERATED_DB>.<DBX_UC_SCHEMA>.*
                    →  MY_DATABRICKS_UC_CI catalog integration
                    →  Databricks Unity Catalog  (catalog-linked database)
```

**Parameters**: Set in the first code cell below.

**API routing in SCOS:**

| API | Use for |
|-----|----------|
| `sf_session.sql(...).to_pandas()` | Snowflake-specific: `CURRENT_ROLE()`, `SHOW`, DDL |
| `switch_role(role)` | `USE ROLE` — must go through sf_session |
| `spark.sql(f"SELECT * FROM {TBL_ORDERS}")` | **Catalog-linked DB tables** (lowercase schema required) |
| `spark.table(...)` | Native Snowflake tables only (not catalog-linked) |


## Step 1 — Initialize Sessions

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake import snowpark_connect
from pyspark import SparkConf
from pyspark.sql.functions import col, lit, count, sum as spark_sum

sf_session = get_active_session()

# Init SCOS on a non-catalog-linked database; CLD as default breaks table resolution
sf_session.sql(f"USE DATABASE {SF_INIT_DB}").collect()

# caseSensitive=true preserves lowercase CLD identifiers (no auto-uppercase)
# so SCOS sends "my_demo_schema"."customer_orders" not "MY_DEMO_SCHEMA"
conf = SparkConf().set("spark.sql.caseSensitive", "true")
spark = snowpark_connect.init_spark_session(conf=conf)

def switch_role(role: str):
    """USE ROLE via sf_session — spark.sql('USE ROLE ...') raises parse error in SCOS."""
    sf_session.sql(f"USE ROLE {role}").collect()
    print(f"Active role → {role}")

switch_role("ACCOUNTADMIN")
print("Sessions ready")


## Step 2 — Verify Session and Federated Tables

In [ ]:
# CURRENT_ROLE / CURRENT_WAREHOUSE are Snowflake-specific → sf_session
# spark.sql('SELECT CURRENT_ROLE()') raises error 4001 in SCOS

ctx = sf_session.sql(
    "SELECT CURRENT_ROLE() AS role, CURRENT_WAREHOUSE() AS wh, "
    "CURRENT_DATABASE() AS db, CURRENT_SCHEMA() AS schema"
).to_pandas()
print("Session context:")
print(ctx.to_string(index=False))



---
## Demo 1 — CUSTOMER_ORDERS: Open Table  `[DataFrame]`
No governance policy. All roles see identical data.

In [ ]:
# 3-part lower-cased name: CLD schema/tables are lowercase in Databricks UC
df_orders = spark.sql(f"SELECT * FROM {TBL_ORDERS}")

print(f"[DataFrame] CUSTOMER_ORDERS — {{df_orders.count()}} rows")
df_orders.printSchema()
df_orders.orderBy("order_id").show(truncate=False)


## Demo 2 — Filter + Select  `[DataFrame]`

In [ ]:
print("[DataFrame] .filter().select() — SHIPPED / DELIVERED only:")
(
    df_orders
    .filter(col("status").isin("SHIPPED", "DELIVERED"))
    .select("order_id", "product", "amount", "status")
    .orderBy("amount", ascending=False)
    .show(truncate=False)
)

## Demo 3 — Aggregation  `[DataFrame]`

In [ ]:
print("[DataFrame] .groupBy().agg() — revenue by status:")
(
    df_orders
    .groupBy("status")
    .agg(
        count("*").alias("order_count"),
        spark_sum("amount").alias("total_revenue")
    )
    .orderBy("total_revenue", ascending=False)
    .show(truncate=False)
)

---
## Demo 4 — SENSITIVE_ORDERS: ACCOUNTADMIN (Unmasked)  `[DataFrame]`

Masking policy: `CURRENT_ROLE() = 'ACCOUNTADMIN'` → real card number.

> Databricks UC column masks do **not** propagate to Snowflake.  
> Snowflake enforces its own independently-defined masking policy.

In [ ]:
switch_role("ACCOUNTADMIN")

print("[DataFrame] SENSITIVE_ORDERS — ACCOUNTADMIN (credit_card unmasked):")
(
    spark.sql(f"SELECT * FROM {TBL_SENSITIVE}")
    .select("order_id", "customer_name", "credit_card", "amount", "region")
    .orderBy("order_id")
    .show(truncate=False)
)


## Demo 5 — SENSITIVE_ORDERS: SNOWFLAKE_READER_ROLE (Masked)  `[DataFrame]`
Masking fires: `credit_card` → `****-****-****-XXXX`

In [ ]:
switch_role(SF_READER_ROLE)

df_sensitive = spark.sql(f"SELECT * FROM {TBL_SENSITIVE}")

print(f"[DataFrame] SENSITIVE_ORDERS — {SF_READER_ROLE} (masked):")
(
    df_sensitive
    .select("order_id", "customer_name", "credit_card", "amount", "region")
    .orderBy("order_id")
    .show(truncate=False)
)

print("[DataFrame] .filter() on masked column:")
(
    df_sensitive
    .filter(col("credit_card").startswith("****"))
    .select("order_id", "credit_card")
    .show(truncate=False)
)


## Demo 6 — Side-by-Side Governance  `[DataFrame union]`
Same Parquet files — different credit_card values per role.

In [ ]:
import pandas as pd

COLS = ["order_id", "customer_name", "credit_card"]

switch_role("ACCOUNTADMIN")
admin_rows = spark.sql(f"SELECT * FROM {TBL_SENSITIVE}").select(*COLS).collect()

switch_role(SF_READER_ROLE)
reader_rows = spark.sql(f"SELECT * FROM {TBL_SENSITIVE}").select(*COLS).collect()

admin_pd  = pd.DataFrame([r.asDict() for r in admin_rows])
admin_pd.insert(0, "role", "ACCOUNTADMIN")

reader_pd = pd.DataFrame([r.asDict() for r in reader_rows])
reader_pd.insert(0, "role", SF_READER_ROLE)

combined = pd.concat([admin_pd, reader_pd]).sort_values(["order_id", "role"]).reset_index(drop=True)
print("[DataFrame eager collect] Same Parquet files — different governance outcome:")
print(combined.to_string(index=False))


## Demo 7 — Join Open + Sensitive Tables  `[DataFrame]`
ACCOUNTADMIN: credit_card unmasked in join result.

In [ ]:
switch_role("ACCOUNTADMIN")

df_open = spark.sql(f"SELECT * FROM {TBL_ORDERS}")
df_prot = spark.sql(f"SELECT * FROM {TBL_SENSITIVE}")

print("[DataFrame .join()] ACCOUNTADMIN — credit_card unmasked:")
(
    df_open.alias("o")
    .join(df_prot.alias("s"),
          col("o.customer_id") == col("s.customer_id"), "inner")
    .select(col("o.order_id"), col("o.product"),
            col("o.amount").alias("product_amount"),
            col("s.customer_name"), col("s.credit_card"), col("s.region"))
    .orderBy("order_id")
    .show(truncate=False)
)


## Demo 8 — Same Join as Reader Role  `[DataFrame]`
SNOWFLAKE_READER_ROLE: masking applied inside the join.

In [ ]:
switch_role("SNOWFLAKE_READER_ROLE")

df_open = spark.sql(f"SELECT * FROM {TBL_ORDERS}")
df_prot = spark.sql(f"SELECT * FROM {TBL_SENSITIVE}")

print("[DataFrame .join()] SNOWFLAKE_READER_ROLE — credit_card masked:")
(
    df_open.alias("o")
    .join(df_prot.alias("s"),
          col("o.customer_id") == col("s.customer_id"), "inner")
    .select(col("o.order_id"), col("o.product"),
            col("o.amount").alias("product_amount"),
            col("s.customer_name"), col("s.credit_card"), col("s.region"))
    .orderBy("order_id")
    .show(truncate=False)
)


## Demo 9 — Sync Catalog-Linked Database  `[Snowpark DDL]`
Catalog-linked databases auto-sync (default: every 30 s).  
`RESUME DISCOVERY` triggers an immediate re-poll of the remote catalog.


In [ ]:
switch_role("ACCOUNTADMIN")

# Catalog-linked databases auto-sync every 30 s.
# RESUME DISCOVERY forces an immediate re-poll of the remote catalog.
sf_session.sql(f"ALTER DATABASE {SF_FEDERATED_DB} RESUME DISCOVERY").collect()

status = sf_session.sql(
    f"SELECT SYSTEM$CATALOG_LINK_STATUS('{SF_FEDERATED_DB}')"
).collect()[0][0]
print(f"Catalog sync status: {status}")
print("CLD discovery resumed ✅  Snowflake polling Databricks UC for latest tables")


---
## Summary

| Demo | API | Role | Result |
|------|-----|------|--------|
| 1 — Read open table | `spark.table().show()` | ACCOUNTADMIN | 5 rows |
| 2 — Filter + select | `df.filter().select()` | ACCOUNTADMIN | SHIPPED/DELIVERED |
| 3 — Aggregation | `df.groupBy().agg()` | ACCOUNTADMIN | Revenue by status |
| 4 — Masked read | `spark.table()` | ACCOUNTADMIN | `4111-1111-1111-1111` |
| 5 — Masked read | `spark.table()` | SNOWFLAKE_READER_ROLE | `****-****-****-1111` |
| 6 — Side-by-side | `admin_df.union(reader_df)` | Both | Governance comparison |
| 7 — Join (admin) | `df_open.join(df_prot)` | ACCOUNTADMIN | credit_card unmasked |
| 8 — Join (reader) | `df_open.join(df_prot)` | SNOWFLAKE_READER_ROLE | credit_card masked |
| 9 — Refresh | `sf_session.sql(ALTER ICEBERG TABLE REFRESH)` | ACCOUNTADMIN | Sync from Databricks |

**Masking is enforced by Snowflake's engine** — applies equally to  
`.filter()`, `.select()`, `.join()`, `.union()` in the SCOS DataFrame API.  
The PySpark layer never receives unmasked values for restricted roles.